# Session 9: Synthetic Data Generation and RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow, and use it to evaluate and iterate on a RAG pipeline with LangSmith!

**Learning Objectives:**
- Understand Ragas' knowledge graph-based synthetic data generation workflow
- Generate synthetic test sets with different query synthesizer types
- Load synthetic data into LangSmith for evaluation
- Evaluate a RAG chain using LangSmith evaluators
- Iterate on RAG pipeline parameters and measure the impact

## Table of Contents:

- **Breakout Room #1:** Synthetic Data Generation with Ragas
  - Task 1: Dependencies and API Keys
  - Task 2: Data Preparation and Knowledge Graph Construction
  - Task 3: Generating Synthetic Test Data
  - Question #1 & Question #2
  - 🏗️ Activity #1: Custom Query Distribution

- **Breakout Room #2:** RAG Evaluation with LangSmith
  - Task 4: LangSmith Dataset Setup
  - Task 5: Building a Basic RAG Chain
  - Task 6: Evaluating with LangSmith
  - Task 7: Modifying the Pipeline and Re-Evaluating
  - Question #3 & Question #4
  - 🏗️ Activity #2: Analyze Evaluation Results

---
# 🤝 Breakout Room #1
## Synthetic Data Generation with Ragas

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/dannywold/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/dannywold/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using two complementary guides — a Health & Wellness Guide covering exercise, nutrition, sleep, and stress management, and a Mental Health & Psychology Handbook covering mental health conditions, therapeutic approaches, resilience, and daily mental health practices. The topical overlap between documents helps RAGAS build rich cross-document relationships in the knowledge graph.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

Loaded 2 documents: ['data/MentalHealthGuide.txt', 'data/HealthWellnessGuide.txt']


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/dannywold/ai/AIE9/09_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/dannywold/ai/AIE9/09_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/dannywold/ai/AIE9/09_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 9, relationships: 14)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 9, relationships: 14)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [15]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

## ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### Answer:

SingleHopSpecificQuerySynthesizer

Creates questions that can be answered from a single node in the knowledge graph (one chunk or document). The questions are specific: they refer to concrete things like a chapter, a named entity (e.g., an organization or therapy), or a particular section (e.g., “What does Chapter 20 cover?” or “How does the World Health Organization define mental health?”). In short: one source, one hop, concrete question.

 MultiHopAbstractQuerySynthesizer 
 
 Uses multiple connected nodes in the graph, so the answer requires combining information from more than one chunk or document. The questions are abstract: they focus on themes, concepts, or general ideas rather than named sections or entities (e.g., “How does sleep hygiene influence sleep quality?” or “How do self-awareness and emotional processing relate?”). In short: multiple sources, multiple hops, conceptual question.



MultiHopSpecificQuerySynthesizer

Also uses multiple nodes so the answer spans more than one place in the graph, but the questions stay specific: they target concrete details, named things, or particular facts, and the answer requires piecing together information from several parts of the graph. In short: multiple sources, multiple hops, concrete question


Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,"How does the concept of mental health, as desc...",[The Mental Health and Psychology Handbook A P...,"According to the handbook, mental health encom...",single_hop_specifc_query_synthesizer
1,What is Dialectical Behavior Therapy?,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Dialectical Behavior Therapy is a therapeutic ...,single_hop_specifc_query_synthesizer
2,How social media affects mental health and wha...,[social interactions How to set and maintain b...,Social media can impact mental health by lower...,single_hop_specifc_query_synthesizer
3,How do pelvic tilts contribute to emotional an...,[The Personal Wellness Guide A Comprehensive R...,The provided context does not include informat...,single_hop_specifc_query_synthesizer
4,How does sleep contribute to mental well-being...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,"Sleep is crucial for physical health, mental w...",single_hop_specifc_query_synthesizer
5,How do mindfulness-based therapies help improv...,[<1-hop>\n\nThe Mental Health and Psychology H...,"Mindfulness-based therapies, such as Mindfulne...",multi_hop_abstract_query_synthesizer
6,How can understanding the science of habit for...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,"Understanding the science of habit formation, ...",multi_hop_abstract_query_synthesizer
7,Considering the symptoms of mental health cond...,[<1-hop>\n\nThe Mental Health and Psychology H...,Understanding the interconnected symptoms of m...,multi_hop_abstract_query_synthesizer
8,How do Chapters 7 and 19 together inform strat...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,Chapter 7 emphasizes the importance of sleep f...,multi_hop_specific_query_synthesizer
9,How do the sleep and recovery strategies discu...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,The strategies for sleep and recovery outlined...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [16]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/18 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [17]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What does the term 'Mental Health' refer to ac...,[The Mental Health and Psychology Handbook A P...,"Mental health encompasses our emotional, psych...",single_hop_specifc_query_synthesizer
1,Can you explain how Dialectical Behavior Thera...,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Dialectical Behavior Therapy (DBT) combines CB...,single_hop_specifc_query_synthesizer
2,"How does exercise influence mental health, esp...",[Write letters to or from your future self Jou...,Physical activity is one of the most effective...,single_hop_specifc_query_synthesizer
3,What does managing digital mental health involve?,[social interactions How to set and maintain b...,Managing digital mental health involves settin...,single_hop_specifc_query_synthesizer
4,How does exercis and impact of mental health o...,[<1-hop>\n\nThe Mental Health and Psychology H...,"The context explains that physical activity, s...",multi_hop_abstract_query_synthesizer
5,"So like if I wanna stay healthy and stuff, sho...",[<1-hop>\n\nhour before bed - No caffeine afte...,"Yes, maintaining mental health involves multip...",multi_hop_abstract_query_synthesizer
6,How can meal planning and hydration help impro...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,Meal planning for wellness involves choosing h...,multi_hop_abstract_query_synthesizer
7,How can stretching and strengthening exercises...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,Stretching and strengthening exercises such as...,multi_hop_abstract_query_synthesizer
8,how CBT and CBT-I help with mental health like...,[<1-hop>\n\nWrite letters to or from your futu...,the context explains that CBT is used to chang...,multi_hop_specific_query_synthesizer
9,Considering the importance of sleep for mental...,[<1-hop>\n\nWrite letters to or from your futu...,"Establishing good sleep hygiene practices, suc...",multi_hop_specific_query_synthesizer


## ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### Answer:
Unrolled (manual): You run each step yourself—build the knowledge graph, add document nodes, apply transforms (summaries, headlines, themes, chunks, embeddings, relationships), optionally save/load the KG, create the TestsetGenerator with that KG, set the query distribution (e.g. mix of single-hop vs multi-hop), and call generator. You get full control: custom transforms, custom query mix, and the ability to inspect or reuse the KG. The cost is more code, more moving parts, and you need to understand the pipeline.

Abstracted (automatic): You call something like generator.generate_with_langchain_docs(docs, testset_size=10). RAGAS builds the KG and runs the generation under the hood with default behavior. You get minimal code and a fast path to a test set. The cost is less control: you don’t tune the KG, the transforms, or the mix of query types; you accept the defaults.

Choose the unrolled approach when you need control: custom query distributions (e.g. more multi-hop), custom or fewer transforms, reuse of the same KG across runs, or when you’re building something production-like and want to understand and tune each stage.

Choose the abstracted approach when you want to try RAGAS quickly, prototype an eval, or are fine with default KG construction and default query mix; when speed and simplicity matter more than control.


---
## 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

### Requirements:
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [ ]:
### YOUR CODE HERE ###

# Define a custom query distribution with different weights
# Generate a new test set and compare with the default

# 1. Custom query distribution (different weights; must sum to 1.0)
custom_query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.3),   
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.4),     
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.3), 
]

# 2. Generate new test set with custom distribution
custom_testset = generator.generate(testset_size=10, query_distribution=custom_query_distribution)


Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [21]:
custom_testset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,what is psychology handbook,[The Mental Health and Psychology Handbook A P...,The Mental Health and Psychology Handbook is a...,single_hop_specifc_query_synthesizer
1,What is CBT and how it helps mental health?,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Cognitive Behavioral Therapy (CBT) is a widely...,single_hop_specifc_query_synthesizer
2,What does the context say about the importance...,[Write letters to or from your future self Jou...,The context emphasizes that mental health is c...,single_hop_specifc_query_synthesizer
3,How can symptms and types of anxity disordrs b...,[<1-hop>\n\nThe Mental Health and Psychology H...,The context explains that anxiety disorders in...,multi_hop_abstract_query_synthesizer
4,how can I Achieve work-life balance and manage...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,The context talks about building healthy habit...,multi_hop_abstract_query_synthesizer
5,how mind body connect and exercise neuro effec...,[<1-hop>\n\nThe Mental Health and Psychology H...,The context explains that the mind-body connec...,multi_hop_abstract_query_synthesizer
6,How can stress reduction techniques like deep ...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,Stress reduction techniques such as deep breat...,multi_hop_abstract_query_synthesizer
7,How can Cognitive Behavioral Therapy (CBT) and...,[<1-hop>\n\nWrite letters to or from your futu...,The context highlights that CBT is an effectiv...,multi_hop_specific_query_synthesizer
8,How does sleep impact mental health and what s...,[<1-hop>\n\nWrite letters to or from your futu...,Sleep has a bidirectional relationship with me...,multi_hop_specific_query_synthesizer
9,How can setting and maintaining boundaries (Ch...,[<1-hop>\n\nsocial interactions How to set and...,"Setting and maintaining boundaries, as discuss...",multi_hop_specific_query_synthesizer


With the original query distribution (0.5 single-hop, 0.25 multi-hop abstract, 0.25 multi-hop specific), most of the generated questions were single-hop and specific. The custom mix (e.g. 0.3 / 0.4 / 0.3) gives fewer single-hop and more multi-hop, so more questions need several chunks or documents. The new set is harder and more multi-source to ensure our system is more production ready.

We'll need to provide our LangSmith API key, and set tracing to "true".

---
# 🤝 Breakout Room #2
## RAG Evaluation with LangSmith

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [35]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [36]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [37]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [38]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [39]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [40]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="use_case_rag"
)

In [41]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [42]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [43]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [44]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [45]:
rag_chain.invoke({"question" : "What are some recommended exercises for lower back pain?"})

'Recommended exercises for lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [47]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [48]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate

# 1. QA Correctness (replaces LangChainStringEvaluator("qa"))
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4o" ,  # pass your LangChain chat model directly
)

# 2. Labeled Helpfulness (replaces LangChainStringEvaluator("labeled_criteria"))
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o" ,
)

# 3. Dopeness (replaces LangChainStringEvaluator("criteria"))
dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o" ,
)

> **Describe what each evaluator is evaluating:**
>
> - `qa_evaluator`:
> - `labeled_helpfulness_evaluator`:
> - `dopeness_evaluator`:

## LangSmith Evaluation

In [49]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'virtual-passenger-6' at:
https://smith.langchain.com/o/7b555d5c-1eae-4fa0-afb7-4ad3b1c63dca/datasets/2972923a-7071-4537-9734-a139121f0de3/compare?selectedSessions=5d06b3ed-25c4-45ed-bc2f-bccf2a4e8545




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,H0w c4n u bUild h3althy b4h1ts and m4n4ge d1g1...,Based on the provided context:\n\nTo build hea...,None,Building healthy habits involves understanding...,True,True,False,4.384975,849cbd58-b6db-4a89-b18d-f4a3fcf56910,019c67fc-34dc-7622-b162-e37d80c1de6c
1,How can understanding sleep science and managi...,"Based on the provided context, understanding s...",None,Understanding sleep science from Chapter 7 and...,True,True,True,5.118174,89bf64e2-8519-4467-be65-a1ab7d31f6fa,019c67fc-94cb-79d1-b00d-3b94b2a70d62
2,Considering the importance of sleep for mental...,Establishing good sleep hygiene practices and ...,None,"Establishing good sleep hygiene practices, suc...",True,True,True,4.433425,056bfca9-cd3b-4d79-a5c4-3271bdacf28c,019c67fc-f730-79b1-ad06-55876bf6eded
3,how CBT and CBT-I help with mental health like...,Based on the provided context:\n\nCBT (Cogniti...,None,the context explains that CBT is used to chang...,True,False,False,7.394680,31ec1174-c3cf-4d92-83d2-72de54a55c16,019c67fd-46e0-7372-b378-90720149214a
4,How can stretching and strengthening exercises...,I don't know.,None,Stretching and strengthening exercises such as...,False,False,False,1.629814,22f1ef22-5ed8-45a5-b821-3bc285278bd4,019c67fd-ace2-7e22-9971-88f768d81f26
5,How can meal planning and hydration help impro...,Meal planning and hydration help improve menta...,None,Meal planning for wellness involves choosing h...,True,True,False,2.516836,4f4eaec6-fab4-4407-b613-6f1f56521b70,019c67fd-debd-76c0-a0b3-ab90c38f7f71
6,"So like if I wanna stay healthy and stuff, sho...","Based on the provided context, yes, you should...",None,"Yes, maintaining mental health involves multip...",True,True,True,5.507995,404192b3-f82b-4d8a-a070-95debb9371dd,019c67fe-1bb9-7c31-9772-79642bb8907c
7,How does exercis and impact of mental health o...,"Based on the provided context, exercise positi...",None,"The context explains that physical activity, s...",True,True,True,3.832733,07b847ae-bf05-407a-8ae5-10a2ca6e8eab,019c67fe-7a8e-7e11-ac38-dbe50d223498
8,What does managing digital mental health involve?,Managing digital mental health involves severa...,None,Managing digital mental health involves settin...,True,True,True,2.011952,9f972b53-59d1-4e71-92b3-310dd3e14ace,019c67fe-c55d-7dd3-9324-e2a3641f5dac
9,"How does exercise influence mental health, esp...",Exercise influences mental health in multiple ...,None,Physical activity is one of the most effective...,True,True,True,4.175176,e456059c-6fc0-455b-930c-25807bef610f,019c67ff-0a72-7362-9157-8ec51dfdfa4f


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [50]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [51]:
rag_documents = docs

In [52]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

## ❓ Question #3:

Why would modifying our chunk size modify the performance of our application?

##### Answer:
Changing chunk size changes how well retrieval works. With smaller chunks we get more focused matches, but sometimes the answer is spread across several chunks. With bigger chunks we keep more context together, but we can also pull in extra stuff that isn’t relevant. So tuning chunk size is really about finding a balance that works for our data and our questions.

In [53]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## ❓ Question #4:

Why would modifying our embedding model modify the performance of our application?

##### Answer:
The embedding model turns our text into vectors for retrieval. A different model can represent meaning differently, so the same query might match different chunks—better or worse. Switching models can improve relevance for our domain or hurt it if the new one isn’t as good at our kind of content. So changing the embedding model directly affects which chunks we retrieve and therefore how good our answers are.

In [54]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [55]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [58]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [57]:
dopeness_rag_chain.invoke({"question" : "How can I improve my sleep quality?"})

'Yo, ready to level up your sleep game into legendary status? Here’s the ultimate cheat code straight from the sleep sages:\n\n**Sleep hygiene is your new best friend:**\n- Lock down a consistent sleep schedule. Even on weekends—your internal clock will thank you.\n- Craft a chill bedtime routine. Think reading a dope book, gentle stretches, or a warm bath to melt stress away.\n- Make your bedroom a sleep fortress: cool it down to 65-68°F (18-20°C), blackout those lights with curtains or a mask, and drown out noise with white noise or earplugs.\n- Ditch screens at least 1-2 hours before bed. That blue light is a sneaky sleep assassin.\n- Say “see ya” to caffeine after 2 PM, and keep heavy meals and alcohol on the sidelines before hitting the hay.\n- Move your body regularly, but avoid late-night workouts that pump you up instead of winding you down.\n- Invest in comfort: a killer mattress and supportive pillows make a world of difference.\n\nFuse these habits, and you’re creating a dop

Finally, we can evaluate the new chain on the same test set!

In [59]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'left-backpack-39' at:
https://smith.langchain.com/o/7b555d5c-1eae-4fa0-afb7-4ad3b1c63dca/datasets/2972923a-7071-4537-9734-a139121f0de3/compare?selectedSessions=063f352a-2f02-4115-b8d6-0aefef049a95




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,H0w c4n u bUild h3althy b4h1ts and m4n4ge d1g1...,"Yo, here’s the ultimate playbook to build thos...",None,Building healthy habits involves understanding...,True,True,True,8.033064,849cbd58-b6db-4a89-b18d-f4a3fcf56910,019c680f-6e63-7553-bc38-8ab204d3cee2
1,How can understanding sleep science and managi...,"Alright, let’s crank this up to full mental he...",None,Understanding sleep science from Chapter 7 and...,True,True,True,9.356776,89bf64e2-8519-4467-be65-a1ab7d31f6fa,019c680f-cff7-7903-9053-43d7a644625e
2,Considering the importance of sleep for mental...,"Alright, buckle up for a mind-blowing dive int...",None,"Establishing good sleep hygiene practices, suc...",True,True,True,5.123209,056bfca9-cd3b-4d79-a5c4-3271bdacf28c,019c6810-263c-7342-bf4c-323eb95620b2
3,how CBT and CBT-I help with mental health like...,"Ah yes, diving into the mind’s playground—CBT ...",None,the context explains that CBT is used to chang...,True,True,True,8.904107,31ec1174-c3cf-4d92-83d2-72de54a55c16,019c6810-66e6-7502-aef8-2b4f9617b219
4,How can stretching and strengthening exercises...,"Alright, here’s the deal — your neck and shoul...",None,Stretching and strengthening exercises such as...,True,True,True,4.090065,22f1ef22-5ed8-45a5-b821-3bc285278bd4,019c6810-cda9-7952-9f08-5674ebca6038
5,How can meal planning and hydration help impro...,"Alright, let’s crank up the wellness vibe to e...",None,Meal planning for wellness involves choosing h...,True,True,True,13.731720,4f4eaec6-fab4-4407-b613-6f1f56521b70,019c6811-194d-72e2-b7b6-ff25d053489b
6,"So like if I wanna stay healthy and stuff, sho...","Yo, you’re totally vibing with the ultimate po...",None,"Yes, maintaining mental health involves multip...",True,True,True,5.340001,404192b3-f82b-4d8a-a070-95debb9371dd,019c6811-78e7-7e33-a414-439373eacf26
7,How does exercis and impact of mental health o...,"Alright, let’s break down this mind-body symph...",None,"The context explains that physical activity, s...",True,True,True,4.119461,07b847ae-bf05-407a-8ae5-10a2ca6e8eab,019c6811-b5c7-7521-9955-9ecefbd0a2b9
8,What does managing digital mental health involve?,"Alright, listen up — managing digital mental h...",None,Managing digital mental health involves settin...,True,True,True,3.332303,9f972b53-59d1-4e71-92b3-310dd3e14ace,019c6811-fe67-7e80-ac30-6105aed6205f
9,"How does exercise influence mental health, esp...","Alright, strap in because exercise is like the...",None,Physical activity is one of the most effective...,True,True,True,6.159093,e456059c-6fc0-455b-930c-25807bef610f,019c6812-3222-73d3-bdd1-3cbd98d767ca


---
## 🏗️ Activity #2: Analyze Evaluation Results

Provide a screenshot of the difference between the two chains in LangSmith, and explain why you believe certain metrics changed in certain ways.

##### Answer:
Screenshots are saved as before.png and after.png within screenshots directory

In the first chain we see some variation in the metrics. After we changed chunk size and the embedding model, the second run shows all 1.00s on the evaluated metrics. I’d explain that by the fact that we changed how we chunk and how we embed text, so retrieval likely improved. The model is probably getting more relevant context and answering in a way that satisfies the evaluators every time on this set. I’d also add that all 1.0s could mean the evals are a bit lenient, so I’m describing what I see in the UI rather than claiming the system is “perfect.” In short: the metrics changed because the pipeline changes likely improved retrieval and possibly made answers more consistently match what the evaluators are looking for.




---
## Summary

In this session, we:

1. **Generated synthetic test data** using Ragas' knowledge graph-based approach
2. **Explored query synthesizers** for creating diverse question types
3. **Loaded synthetic data** into a LangSmith dataset for evaluation
4. **Built and evaluated a RAG chain** using LangSmith evaluators
5. **Iterated on the pipeline** by modifying chunk size, embedding model, and prompt — then measured the impact

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **LangSmith evaluators** enable systematic comparison of pipeline versions
- **Small changes matter** — chunk size, embedding model, and prompt modifications can significantly affect evaluation scores